# Fail App Code

## Pre-Reqs

### Create df

**Output:** df_deployments

In [1]:
import pandas as pd

# Useful reference: <https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html>
raw_file_path = '/home/lnx_workspaces/bpp_projects/bpp_module2_project/doraview/data/json/deployment_frequency.json'

# Try reading with explicit encoding and error handling
try:
    df_deployments = pd.read_json(raw_file_path, encoding='utf-8', convert_dates=["deployed_at"])
    print("\nDataframe info:")
    print(df_deployments.info())
    print("\nFirst 5 rows:")
    print(df_deployments.head())
except Exception as e:
    print(f"Error: {str(e)}")

df_deployments["month"] = df_deployments["deployed_at"].dt.month

df_deployments.head(100)


Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   deployment_id     44 non-null     object             
 1   application_id    44 non-null     object             
 2   application_name  44 non-null     object             
 3   environment       44 non-null     object             
 4   deployed_at       44 non-null     datetime64[ns, UTC]
 5   status            44 non-null     object             
 6   version           44 non-null     object             
dtypes: datetime64[ns, UTC](1), object(6)
memory usage: 2.5+ KB
None

First 5 rows:
  deployment_id application_id application_name environment  \
0   df_55eeff30         app001     Auth Service  production   
1   df_952faf50         app001     Auth Service     staging   
2   df_45056a55         app001     Auth Service  production   
3   df_4a4be47

,deployment_id,application_id,application_name,environment,deployed_at,status,version,month
0,df_55eeff30,app001,Auth Service,production,2025-08-31 02:05:00+00:00,success,v3.3.4,8
1,df_952faf50,app001,Auth Service,staging,2025-08-07 11:06:00+00:00,success,v3.7.0,8
2,df_45056a55,app001,Auth Service,production,2025-07-27 12:08:00+00:00,success,v2.0.3,7
3,df_4a4be475,app001,Auth Service,staging,2025-08-16 06:24:00+00:00,success,v3.8.4,8
4,df_9ddc9087,app001,Auth Service,staging,2025-07-29 03:29:00+00:00,success,v1.7.6,7
5,df_09f6a4a8,app001,Auth Service,production,2025-09-30 17:01:00+00:00,success,v1.3.6,9
6,df_0721d117,app001,Auth Service,production,2025-08-11 14:09:00+00:00,success,v2.1.9,8
7,df_3ad908f6,app001,Auth Service,staging,2025-08-07 03:03:00+00:00,success,v3.9.3,8
8,df_295ed3f1,app001,Auth Service,production,2025-07-07 16:35:00+00:00,success,v2.4.3,7
9,df_1df285b7,app001,Auth Service,staging,2025-08-09 09:38:00+00:00,success,v2.9.6,8


## Calculate

### Group count by status type

**Output:** df_status_grouped

In [2]:
target_app_id = 'app002'

In [3]:
df_target = df_deployments[df_deployments['application_id'] == target_app_id]

In [4]:
df_status_grouped = df_target.groupby(['month', 'status']).agg({'status':'count'})

print(df_status_grouped)

               status
month status         
7     failed        1
      success       1
8     failed        1
      success       4
9     failed        1
      success       3


### Calculate status counts as percentages

**Output:** df_status_percent

In [ ]:
df_status_percent = df_status_grouped.groupby(level=0).apply(
	lambda x: 100 * x / x.sum())

# Fix the index (drop the duplicate month level)
df_status_percent.index = df_status_percent.index.droplevel(1)

# Rename the column to avoid conflict during reset_index
df_status_percent = df_status_percent.rename(columns={'status':'percentage'})

# Reset index to create a DataFrame suitable for plotting
df_status_percent = df_status_percent.reset_index()

print(df_status_percent)

df_status_percent = 
               percentage
month status             
7     failed         50.0
      success        50.0
8     failed         20.0
      success        80.0
9     failed         25.0
      success        75.0
Columns =  Index(['percentage'], dtype='object')
Index names =  ['month', 'status']
   month   status  percentage
0      7   failed        50.0
1      7  success        50.0
2      8   failed        20.0
3      8  success        80.0
4      9   failed        25.0
5      9  success        75.0


## Create graph

In [7]:
# from dash import dcc
# import matplotlib.pyplot as plt
import plotly.express as px

In [10]:
# display dataframe as figure
fig_month_stat_bar = px.bar(
	data_frame=df_status_percent,
	title="Deployment Failure Rates by Month",
	x="month",
	y="percentage",
	color="status",
	color_discrete_map={
		"success":"#636EFA",
		"failed":"#EF553B"
		},
)

fig_month_stat_bar.update_layout(barmode='stack')

fig_month_stat_bar.update_yaxes(
	title_text="Percentage (%)"
)

fig_month_stat_bar.update_xaxes(
	title_text="Month",
	tickvals=list(range(1,13)),
	ticktext=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
)

fig_month_stat_bar.show()